In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("yaml") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyyaml"])

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import Image as DisplayImage, display
cwd = Path.cwd().resolve()
candidates = [cwd / "domain-adaptation", cwd, cwd.parent]
candidates += list(cwd.glob("*/domain-adaptation"))
TASK_DIR = next((p for p in candidates if p.exists() and p.name == "domain-adaptation"), cwd / "domain-adaptation")
REPO_ROOT = TASK_DIR.parent
for path in [TASK_DIR, REPO_ROOT]:
    if str(path) not in sys.path: sys.path.insert(0, str(path))
from shared.pacs import find_pacs_root
from train import train_suite
from evaluate_final import evaluate_suite
PACS_ROOT = None
resolved_pacs_root = find_pacs_root(PACS_ROOT)
print({"task_dir": str(TASK_DIR), "pacs_root": str(resolved_pacs_root)})

In [ ]:
training = train_suite(resolved_pacs_root, TASK_DIR, include_study=True, force=False)
display(training["run_manifest"])
display(training["histories"].groupby("method").tail(1).round(4))
display(DisplayImage(filename=str(TASK_DIR / "results/main_training_curves.png"), width=1050))
display(DisplayImage(filename=str(TASK_DIR / "results/dann_strength_training_curves.png"), width=1050))

In [ ]:
CONFIGURATIONS_LOCKED = False
assert CONFIGURATIONS_LOCKED, (
    "Paused before loading Sketch labels. Confirm the complete experimental design is locked, "
    "then set CONFIGURATIONS_LOCKED=True and rerun this cell."
)

In [ ]:
final = evaluate_suite(resolved_pacs_root, TASK_DIR)

In [ ]:
display(final["main_comparison"].round(4))
display(DisplayImage(filename=str(TASK_DIR / "results/domain_separability_vs_target_accuracy.png"), width=750))
display(DisplayImage(filename=str(TASK_DIR / "results/target_confusion_matrices.png"), width=1050))

In [ ]:
display(final["per_class"].round(4))
display(DisplayImage(filename=str(TASK_DIR / "results/target_per_class_accuracy_changes.png"), width=1000))
display(DisplayImage(filename=str(TASK_DIR / "results/selected_target_failure_cases.png"), width=1000))

In [ ]:
display(final["study"][["grl_max_strength", "mean_source_accuracy", "mean_source_macro_f1", "target_accuracy", "target_macro_f1", "domain_separability"]].round(4))
display(DisplayImage(filename=str(TASK_DIR / "results/dann_strength_study.png"), width=1000))

In [ ]:
audit = pd.read_json(TASK_DIR / "results/completion_audit.json", typ="series")
display(audit.to_frame("value"))
print("Report-ready files:")
for path in sorted((TASK_DIR / "results").iterdir()):
    if path.is_file() and path.name != ".gitkeep": print(" -", path.name)